# Tiền xử lý (chạy 1 lần cho cả nhóm)

Thu thập 3 dataset → loại trùng (MD5 + dHash 256 bit) → nhóm ảnh gần trùng → chia `split_mixed` /
`split_external` + 5 fold CV trên train → mask Otsu (kênh S) → lưu ảnh 224×224 + `manifest.csv` + `stats.json`.

**Kaggle:** Add Input 3 dataset `anhnguyen7508/bananaid`, `anhnguyen7508/bananaimagebd`,
`kimlunl/bananaripeness-github`; bật Internet. Chạy hết → **Save Version** → ở tab Output chọn
**New Dataset** (vd. tên `banana-preprocessed`) để cả nhóm Add Input dùng chung.

Code: `src/preprocessing/`. Notebook chỉ gọi hàm.

In [ ]:
# === SETUP: chạy cell này đầu tiên (Kaggle / Colab / máy) ===
# Lấy code .py theo thứ tự:
#   1. chạy trên máy trong thư mục repo
#   2. code .py đã UPLOAD lên Kaggle (Add Input -> dataset chứa thư mục src/ + configs/) hoặc Colab (/content)
#   3. git clone từ GitHub (Kaggle: bật Settings -> Internet = On)
import os, sys, shutil, subprocess
from pathlib import Path

CODE_SOURCE = "auto"       # "auto" | "upload" | "git"
REPO_URL = "https://github.com/LuanLuKim/cs231-banana-ripeness.git"
BRANCH = "main"
OWNER = "luan"          # tên người chạy, ghi vào bảng kết quả chung

def find_uploaded_code(roots=("/kaggle/input", "/content", "/content/drive/MyDrive"), max_depth=5):
    """Thư mục gốc của code đã upload: thư mục chứa package src/ (src/__init__.py + src/experiment.py)."""
    skip = {"images", "masks", "results", "cache", "reports", ".git", "drive", "sample_data", "__pycache__"}
    for base in map(Path, roots):
        if not base.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(base):
            d = Path(dirpath)
            if d.name == "src" and "__init__.py" in filenames and "experiment.py" in filenames:
                return d.parent
            depth = len(d.parts) - len(base.parts)
            dirnames[:] = [] if depth >= max_depth else sorted(x for x in dirnames if x not in skip)
    return None

local = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").exists()), None)
uploaded = find_uploaded_code() if CODE_SOURCE in ("auto", "upload") and local is None else None
if local is not None and CODE_SOURCE == "auto":
    CODE_DIR, source = local, "máy"
elif uploaded is not None:
    # chép sang /tmp: thư mục /kaggle/input chỉ đọc, và để output notebook không chứa code
    CODE_DIR = Path("/tmp/cs231-code")
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(uploaded, CODE_DIR, ignore=shutil.ignore_patterns("__pycache__", ".git", "*.ipynb"))
    source = f"upload ({uploaded})"
elif CODE_SOURCE == "upload":
    raise FileNotFoundError("Không thấy code đã upload: Add Input dataset chứa thư mục src/ và configs/")
else:
    CODE_DIR = Path("/tmp/cs231-banana-ripeness")
    if (CODE_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
    commit = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    source = f"git {commit}"
sys.path.insert(0, str(CODE_DIR))      # thư mục gốc repo -> import src.*
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]        # chạy lại cell sau khi đổi code -> nạp bản mới
os.environ["BANANA_OWNER"] = OWNER
print("Code:", CODE_DIR, "| nguồn:", source)
if not (CODE_DIR / "configs").exists():
    print("!! Thiếu thư mục configs/ (cần configs/kaggle.yaml, configs/experiments.yaml) — upload cả thư mục configs")

%load_ext autoreload
%autoreload 2

In [ ]:
from src import load_config
from src.preprocessing import run_preprocessing

cfg = load_config(CODE_DIR / "configs" / "kaggle.yaml")   # đường dẫn 3 dataset nằm trong file này
# Nếu Kaggle gắn dataset ở đường dẫn khác, sửa: cfg.dataset_dirs["BananaID"] = "/kaggle/input/..."
for k, v in cfg.dataset_dirs.items():
    print(f"{k:24s} {v}  {'OK' if Path(v).exists() else '!! KHÔNG THẤY'}")

In [ ]:
result = run_preprocessing(cfg)

## Bảng kiểm tra

In [ ]:
for name in ["eda_check", "summary_before_after", "dedup_log", "hash64_vs_256", "split_mixed",
             "split_external", "split_external_dataset", "leakage", "hue_masked", "mask_stats"]:
    if name in result.tables:
        print(f"\n===== {name} =====")
        display(result.tables[name])

## Hình QA (mask Otsu, cặp ảnh nghi trùng khác nhãn, so sánh hash 64 vs 256 bit)

In [ ]:
from IPython.display import Image, display
for name, path in result.figures.items():
    print(name, path)
    display(Image(filename=str(path)))

## Cảnh báo cần báo nhóm

In [ ]:
for w in result.warnings:
    print("-", w)
print("Output:", result.output_dir)